# Sales Data Analysis: Exploring Sales Trends & Business Insights
**Tools:** Python · pandas · matplotlib

This notebook analyses an Indian e-commerce sales dataset (2018) made of two files:

| File | Contents |
|---|---|
| `Orders.csv` | One row per order: Order ID, Order Date, CustomerName, State, City |
| `Details.csv` | One row per product in an order: Order ID, Amount, Profit, Quantity, Category, Sub-Category, PaymentMode |

**Business questions**
1. Which categories and sub-categories give the most sales and profit?
2. Which states and customers contribute the most?

**Workflow:** load data → check quality → merge → `groupby` summaries → visualise (see `02_visualization_dashboard.ipynb`).

## 1. Load the data

In [1]:
#Reading both files 
import pandas as pd

details = pd.read_csv('Details.csv')
orders = pd.read_csv('Orders.csv')


## 2. Data quality check
Inspect data types, non-null counts and missing values in both files before merging.

In [2]:
# Checking data types and non-null values 
print("\nDetails Information:")
details.info()
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print("\nOrders Information:")
orders.info()
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print("Missing values in Details:")
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print(details.isnull().sum())
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print("Missing values in Details:")
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print(orders.isnull().sum())


Details Information:
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Order ID      1500 non-null   str  
 1   Amount        1500 non-null   int64
 2   Profit        1500 non-null   int64
 3   Quantity      1500 non-null   int64
 4   Category      1500 non-null   str  
 5   Sub-Category  1500 non-null   str  
 6   PaymentMode   1500 non-null   str  
dtypes: int64(3), str(4)
memory usage: 123.6 KB
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Orders Information:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Order ID      500 non-null    str  
 1   Order Date    500 non-null    str  
 2   CustomerName  500 non-null    str  
 3   State         500 non-null    str  
 4   City          500 non-null    str  
dtypes:

The data does not have null values , So we moving forward to merge both files 

## 3. Merge both files
Join the two tables on the common key `Order ID`, so each product row also carries its order date, customer and location.

In [3]:
Final_df = pd.merge(
    details,
    orders,
    on='Order ID',
    how='outer',
    )
display(Final_df)

,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order Date,CustomerName,State,City
0,B-25601,80,-56,4,Electronics,Electronic Games,UPI,01-04-2018,Bharat,Gujarat,Ahmedabad
1,B-25601,66,-12,5,Clothing,Stole,UPI,01-04-2018,Bharat,Gujarat,Ahmedabad
2,B-25601,1275,1148,7,Furniture,Bookcases,EMI,01-04-2018,Bharat,Gujarat,Ahmedabad
3,B-25601,8,-2,3,Clothing,Hankerchief,Credit Card,01-04-2018,Bharat,Gujarat,Ahmedabad
4,B-25602,2617,1151,4,Electronics,Phones,Credit Card,01-04-2018,Vrinda,Maharashtra,Pune
...,...,...,...,...,...,...,...,...,...,...,...
1495,B-26099,2366,552,5,Clothing,Trousers,UPI,30-03-2018,Bhishm,Maharashtra,Mumbai
1496,B-26099,9,3,1,Clothing,Skirt,COD,30-03-2018,Bhishm,Maharashtra,Mumbai
1497,B-26100,828,230,2,Furniture,Chairs,COD,31-03-2018,Hitika,Madhya Pradesh,Indore
1498,B-26100,34,10,2,Clothing,T-shirt,Debit Card,31-03-2018,Hitika,Madhya Pradesh,Indore


Now that all the data has been successfully merged, we will proceed with GroupBy operations. 

## 4. Category and sub-category analysis
### 4.1 Total sales per category
Adds up `Amount` for Clothing, Electronics and Furniture. The result is also saved as a CSV for the charts later.

In [4]:
#Category-wise Sales

category_sales = Final_df.groupby('Category')['Amount'].sum()
print(category_sales)

category_sales.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/category_sales.csv")


Category
Clothing       144323
Electronics    166267
Furniture      127181
Name: Amount, dtype: int64


let's move on to Subcategory analysis.

### 4.2 Sales per sub-category
Same idea one level deeper: sales for every sub-category inside each category.

In [5]:
#Sub-Category wise Sale

sub_category_sales = Final_df.groupby(
    ['Category', 'Sub-Category']
)['Amount'].sum()

print(sub_category_sales)

sub_category_sales.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/sub-category_sales.csv")


Category     Sub-Category    
Clothing     Hankerchief         14294
             Kurti                3361
             Leggings             2106
             Saree               59094
             Shirt                7555
             Skirt                1946
             Stole               18546
             T-shirt              7382
             Trousers            30039
Electronics  Accessories         21728
             Electronic Games    39168
             Phones              46119
             Printers            59252
Furniture    Bookcases           56861
             Chairs              34222
             Furnishings         13484
             Tables              22614
Name: Amount, dtype: int64


### 4.3 Profit per sub-category
Same grouping, but summing `Profit` instead of `Amount`, to see which products earn the most.

In [6]:
Profit_sub_category_sales = Final_df.groupby(
    ['Category', 'Sub-Category']
)['Profit'].sum()

print(Profit_sub_category_sales)

Profit_sub_category_sales.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/Profit_sub-category_sales.csv")



Category     Sub-Category    
Clothing     Hankerchief         1823
             Kurti               -401
             Leggings            -130
             Saree               4057
             Shirt               1513
             Skirt               -315
             Stole               2431
             T-shirt             1500
             Trousers            2847
Electronics  Accessories         3353
             Electronic Games    -644
             Phones              1847
             Printers            8606
Furniture    Bookcases           6516
             Chairs              1627
             Furnishings         -806
             Tables              3139
Name: Profit, dtype: int64


Filter every category and group by Sub-Category with respect to profit 

### 4.4 Profit by sub-category, one category at a time
Each cell filters one category and ranks its sub-categories from highest to lowest profit.

In [7]:
Clothing_profit = Final_df[Final_df['Category'] == 'Clothing'].groupby(
    'Sub-Category'
)['Profit'].sum().sort_values(ascending=False)

print(Clothing_profit)

Clothing_profit.to_csv("Clothing_profit.csv")


Sub-Category
Saree          4057
Trousers       2847
Stole          2431
Hankerchief    1823
Shirt          1513
T-shirt        1500
Leggings       -130
Skirt          -315
Kurti          -401
Name: Profit, dtype: int64


In [8]:

Electronics_profit = Final_df[Final_df['Category'] == 'Electronics'].groupby(
    'Sub-Category'
)['Profit'].sum().sort_values(ascending=False)

print(Electronics_profit)

Electronics_profit.to_csv("Electronics_profit.csv")

Sub-Category
Printers            8606
Accessories         3353
Phones              1847
Electronic Games    -644
Name: Profit, dtype: int64


In [9]:
Furniture_profit = Final_df[Final_df['Category'] == 'Furniture'].groupby(
    'Sub-Category'
)['Profit'].sum().sort_values(ascending=False)

print(Furniture_profit)

Furniture_profit.to_csv("Furniture_profit.csv")

Sub-Category
Bookcases      6516
Tables         3139
Chairs         1627
Furnishings    -806
Name: Profit, dtype: int64


Filter every category and group by Sub-Category with respect to Sales

### 4.5 Sales by sub-category, one category at a time
The same three filters again, this time ranking by sales (`Amount`).

In [10]:
Clothing_Sales = Final_df[Final_df['Category'] == 'Clothing'].groupby(
    'Sub-Category'
)['Amount'].sum().sort_values(ascending=False)

print(Clothing_Sales)

Clothing_Sales.to_csv("Clothing_Sales.csv")


Sub-Category
Saree          59094
Trousers       30039
Stole          18546
Hankerchief    14294
Shirt           7555
T-shirt         7382
Kurti           3361
Leggings        2106
Skirt           1946
Name: Amount, dtype: int64


In [11]:
Electronics_Sales = Final_df[Final_df['Category'] == 'Electronics'].groupby(
    'Sub-Category'
)['Amount'].sum().sort_values(ascending=False)

print(Electronics_Sales)

Electronics_Sales.to_csv("Electronics_Sales.csv")

Sub-Category
Printers            59252
Phones              46119
Electronic Games    39168
Accessories         21728
Name: Amount, dtype: int64


In [12]:
Furniture_Sales = Final_df[Final_df['Category'] == 'Furniture'].groupby(
    'Sub-Category'
)['Amount'].sum().sort_values(ascending=False)

print(Furniture_Sales)

Furniture_Sales.to_csv("Furniture_Sales.csv")

Sub-Category
Bookcases      56861
Chairs         34222
Tables         22614
Furnishings    13484
Name: Amount, dtype: int64


Top 10 State by Profit & Sales

## 5. Top 10 states
States ranked by total profit, then by total sales.

In [13]:
State_Profit = Final_df.groupby('State')['Profit'].sum().nlargest(10)
print(State_Profit)

State_Profit.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/State_Profit.csv")

State
Madhya Pradesh      7382
Maharashtra         6963
Uttar Pradesh       3358
Gujarat             3001
Tamil Nadu          2602
Kerala              2435
West Bengal         2074
Delhi               1958
Bihar               1787
Himachal Pradesh    1662
Name: Profit, dtype: int64


In [14]:
State_sales = Final_df.groupby('State')['Amount'].sum().nlargest(10)
print(State_sales)

State_sales.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/State_sales.csv")

State
Maharashtra       102498
Madhya Pradesh     87463
Uttar Pradesh      38362
Delhi              22957
Rajasthan          22334
Gujarat            21371
Punjab             16786
West Bengal        14328
Kerala             13871
Bihar              13417
Name: Amount, dtype: int64


Top 10 customer by profit & Sales

## 6. Top 10 customers
Customers ranked by total profit, then by total sales.

In [15]:
Top_Cust_Profit = Final_df.groupby('CustomerName')['Profit'].sum().nlargest(10)
print(Top_Cust_Profit)
Top_Cust_Profit.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/Top_Cust_Profit.csv")

CustomerName
Madan Mohan    2166
Aarushi        2059
Shrichand      1901
Madhav         1848
Priyanka       1276
Sarita         1183
Bhishm         1165
Bharat         1143
Abhijeet       1078
Pournamasi     1027
Name: Profit, dtype: int64


In [16]:
Top_Cust_Sales = Final_df.groupby('CustomerName')['Amount'].sum().nlargest(10)
print(Top_Cust_Sales)
Top_Cust_Sales.to_csv("/Users/krutikkukde/Desktop/DATA CLEANING/Top_Cust_Sales.csv")


CustomerName
Harivansh      9902
Madhav         9365
Madan Mohan    7766
Shiva          6339
Vishakha       6120
Vrinda         5820
Lalita         5809
Priyanka       5762
Shruti         5750
Abhijeet       5691
Name: Amount, dtype: int64
